# Analyzing Corporate Venturing Activities in Company Disclosures
### <span style="color: blue;"> A hands-on introduction to LLMs and computational text analysis 
##### October 2026, Sheryl Winston Smith

**<span style="color: green;">Research question** How do large companies engage with startups, and why? Firms rarely answer this in one place. The evidence is scattered across annual reports (10-Ks), corporate websites, and press releases, in very different language.

**<span style="color: green;">What is this?** This notebook is a miniature version of one of my real research pipelines. This exercise reads full 10-K filings and company websites for 60 companies and scores each one on 15 dimensions of corporate venturing. Here we use **3 companies and 19 short passages**, so everything runs in seconds and you can see every step.

**<span style="color: green;">What will you learn?**

| Part | Method | Key idea |
|---|---|---|
| 1 | Text as data | Documents are long; LLMs read them in windows |
| 2 | Dictionary (keyword) coding | Fast and transparent, but words are ambiguous |
| 3 | Similarity to definitions (TF-IDF) | Compares wording, not meaning |
| 4 | LLM coding with a prompt | Reads for meaning; must be checked |
| 5 | You as the human coder | Validity: who is right? |
| 6 | From evidence to company scores | Aggregation, comparison, reliability |

<span style="color: blue;">Exercises for later: Look for cells marked **🔧 Try it**. Those are where you change something and re-run.

> <span style="color: green;"> **No API key needed.** Part 4 uses saved LLM answers by default. 
> note: If you were to do this with a live Anthropic API key, set `USE_API = True` to call the model live. We are NOT doing this right now. (The one place a key would go is the 🔑 cell in Part 4.)

### <span style="color: green;">Who is who: passage IDs and companies

<span style="color: blue;">Every passage has an ID. The **letter** tells you the company and the **number** tells you which passage. Merck uses **R** because **M** is already taken by Microsoft.

| Letter | Company | Ticker | Industry |
|---|---|---|---|
| **G** | Alphabet (Google) | GOOGL | Technology |
| **M** | Microsoft | MSFT | Technology |
| **R** | Merck & Co. | MRK | Pharmaceuticals |

| ID | Company | Source | What the passage is about |
|---|---|---|---|
| G1 | Alphabet | 10-K | Other Bets: X (moonshot factory) and Waymo |
| G2 | Alphabet | 10-K | Gains and losses on non-marketable equity (stakes in private companies) |
| G3 | Alphabet | 10-K | Google Cloud business |
| G4 | Alphabet | 10-K | Founders' letter: "not a conventional company" |
| G5 | Alphabet | 10-K | AI infrastructure and "AI accelerator" chips *(keyword trap)* |
| G6 | Alphabet | website | Google for Startups Accelerator |
| M1 | Microsoft | 10-K | Investment and partnership with OpenAI |
| M2 | Microsoft | 10-K | Platform-based ecosystems |
| M3 | Microsoft | 10-K | Sustainability goals (carbon negative by 2030) |
| M4 | Microsoft | 10-K | Cash and equity investments on the balance sheet |
| M5 | Microsoft | 10-K | Goodwill testing and "business climate" *(keyword trap)* |
| M6 | Microsoft | 10-K | Stock compensation, "accelerated method" *(keyword trap)* |
| M7 | Microsoft | website | M12, Microsoft's venture capital fund |
| R1 | Merck | 10-K | Externally sourced programs: licensing and alliances |
| R2 | Merck | 10-K | Restructuring and "accelerated depreciation" *(keyword trap)* |
| R3 | Merck | 10-K | Focus on emerging markets |
| R4 | Merck | 10-K | Competition and R&D |
| R5 | Merck | website | Merck Digital Sciences Studio accelerator |
| R6 | Merck | website | MDSS program features: mentoring, workshops, investment |

<span style="color: blue;">Part 6 also shows full-pipeline scores for three more companies: **Cisco (CSCO)**, **Qualcomm (QCOM)** and **Amazon (AMZN)**.

## 0. Setup

Keep the `data/` folder next to this notebook. Required packages: `pandas`, `matplotlib`, `scikit-learn` (plus `anthropic` only for live LLM calls). In your environment (terminal) run `pip install pandas matplotlib scikit-learn anthropic`.

🔑 **API key (optional):** this notebook does not need one. If you want to call the model live later, there is one clearly marked 🔑 cell in Part 4 where the key goes.

In [ ]:
import json, re, textwrap
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")          # folder that sits next to this notebook
pd.set_option("display.max_colwidth", 120)

definitions = pd.read_csv(DATA / "definitions.csv")
passages    = pd.read_csv(DATA / "passages.csv")
scores      = pd.read_csv(DATA / "pipeline_scores.csv")
cached_llm  = json.loads((DATA / "cached_llm_responses.json").read_text())
score_cards = json.loads((DATA / "pipeline_score_cards.json").read_text())

def show(text, width=100):
    print(textwrap.fill(text, width))

print(f"{len(definitions)} dimensions, {len(passages)} passages, {len(scores)} companies with full-pipeline scores")

## The framework: 15 dimensions in three groups

* **Programs (4)**: *how* a firm engages with startups, e.g. a venture capital fund or an accelerator.
* **Activities (4)**: *what support* it offers, e.g. mentoring or workspace.
* **Objectives (7)**: *why* it does it, e.g. access to new technology or financial return.

These definitions are exactly what the research pipeline gives the LLM.

In [ ]:
labels = {"program": "PROGRAMS: how", "activity": "ACTIVITIES: what support", "objective": "OBJECTIVES: why"}
for group, rows in definitions.groupby("group", sort=False):
    print(f"\n=== {labels[group]} ===")
    for _, r in rows.iterrows():
        print(f"- {r.dimension}")

In [ ]:
# Read one definition in full
show(definitions.set_index("dimension").loc["Venture Clienting", "definition"])

## 1. Text as data

Our data are short passages taken from two kinds of sources:
* **10-K**: the annual report US-listed companies file with the SEC. It is long, legalistic, and written for investors.
* **website**: text scraped from the company's venture or accelerator pages. It is promotional, and scraping picks up noise.

In [ ]:
passages["chars"] = passages.text.str.len()
passages["words"] = passages.text.str.split().str.len()
passages[["passage_id", "ticker", "source", "words", "text"]]

In [ ]:
# Read a passage. 🔧 Try it: change the id (G1 to G6, M1 to M7, R1 to R6)
pid = "G1"
row = passages.set_index("passage_id").loc[pid]
print(pid, "|", row.ticker, "|", row.source, "\n")
show(row.text)

**<span style="color: blue;">**Look closely.** You will see "Alphabets" and "Companys" with no apostrophe. The preprocessing step stripped some characters when it converted the SEC filing to plain text. Every pipeline makes choices like this, and they can matter later, for example when you search for an exact phrase.

### Why windows? The size problem

A single passage here is about 100 words. The full 10-K narratives that the research pipeline reads are far longer:

In [ ]:
# Sizes of the full 10-K narrative text in the research pipeline (characters)
full_10k_chars = {"GOOGL": 253_309, "MSFT": 608_759, "MRK": 508_966}

for t, n in full_10k_chars.items():
    print(f"{t}: {n:>9,} characters  ≈ {n/5.5:>8,.0f} words  ≈ {n/4:>8,.0f} tokens")

**<span style="color: blue;">LLMs read text in **tokens** (roughly 4 characters of English each). The pipeline does not send a whole 10-K at once. It cuts the text into **overlapping windows** so that a sentence near a boundary is never lost, sends each window with the same instructions, and then merges what comes back.

In [ ]:
def make_windows(text_length, window_chars, overlap_chars):
    # Return (start, end) character positions of overlapping windows
    windows, start = [], 0
    while start < text_length:
        end = min(start + window_chars, text_length)
        windows.append((start, end))
        if end == text_length:
            break
        start = end - overlap_chars
    return windows

# The research pipeline uses 60,000-character windows with a 6,000-character overlap
WINDOW, OVERLAP = 60_000, 6_000

for t, n in full_10k_chars.items():
    w = make_windows(n, WINDOW, OVERLAP)
    print(f"{t}: {len(w)} windows → {len(w)} LLM calls just to extract evidence")

**<span style="color: blue;"> ** 🔧 Try it (some exercises for later)** 
    
<span style="color: blue;"> 1. Halve the window to 30,000 characters. How many calls does Microsoft need now?
 2. Set the overlap to 0. What could go wrong with a sentence that straddles two windows?
3. Rough cost: if input costs *p* dollars per million tokens, write one line that estimates the cost of reading all three 10-Ks once. (Check current prices on the provider's website; they change often.)**

In [ ]:
# Your cost estimate here
price_per_million_input_tokens = 3.00   # placeholder, look up the current price
total_tokens = sum(full_10k_chars.values()) / 4
print(f"≈ {total_tokens:,.0f} tokens ≈ ${total_tokens / 1e6 * price_per_million_input_tokens:.2f} per full read")

## 2. Dictionary coding: count the keywords

The oldest computational method: write a list of words for each concept and count matches. It is cheap, fast, and completely transparent.

In [ ]:
# 🔧 Try it: edit these lists, then re-run this cell and the next two
KEYWORDS = {
    "Corporate Venture Capital":   ["venture capital", "invest in", "investor in", "equity securities", "SAFE", "seed"],
    "Accelerators and Incubators": ["accelerat", "incubat", "cohort", "moonshot"],
    "Venture Building":            ["independent companies", "spin-out", "spin out", "new business"],
    "Venture Clienting":           ["partnership", "alliance", "collaboration", "licensing"],
    "Mentorship":                  ["mentor", "advisor", "expert", "guidance"],
    "Events":                      ["workshop", "session", "demo day", "hackathon", "conference"],
    "Access to New Technology":    ["technolog", "innovation", "intellectual property"],
    "Social and Environmental Impact": ["climate", "sustainab", "carbon", "social"],
    "Market Expansion":            ["new markets", "emerging markets", "international", "expand"],
    "Develop Business Ecosystems": ["ecosystem", "network effects", "partners"],
    "Financial Return":            ["gains", "returns", "revenue-sharing", "exits"],
}

def keyword_hits(text):
    # Return {dimension: [matched keywords]} for one passage (case-insensitive)
    hits = {}
    low = text.lower()
    for dim, words in KEYWORDS.items():
        found = [w for w in words if w.lower() in low]
        if found:
            hits[dim] = found
    return hits

passages["keyword_hits"] = passages.text.apply(keyword_hits)
passages["keyword_dims"] = passages.keyword_hits.apply(lambda h: sorted(h))
passages[["passage_id", "ticker", "keyword_hits"]]

In [ ]:
# Which passages did the dictionary flag for "Accelerators and Incubators"?
dim = "Accelerators and Incubators"
for _, r in passages.iterrows():
    if dim in r.keyword_hits:
        m = re.search(r"accelerat\w*|incubat\w*|cohort|moonshot", r.text, re.I)
        snippet = r.text[max(0, m.start()-60): m.end()+60]
        print(f"{r.passage_id}: ...{snippet}...")

<span style="color: blue;">**Go deeper:** Read the matches above. Some are real startup accelerators. Others are "AI accelerator options" (computer chips), "accelerated depreciation" (accounting), or "the accelerated method" (stock compensation). In a dictionary, a word is a word.

<span style="color: blue;">The same thing happens with "climate": Microsoft's goodwill note talks about a change in the "business climate".

<span style="color: blue;">**🔧 Try it**
1. Make the Accelerators list more precise (for example, `"startup accelerator"` or `"accelerator that"`). How many true accelerators do you lose?
2. This trade-off has names: **precision** (of what you flagged, how much is right?) and **recall** (of what is really there, how much did you find?).

## 3. Similarity to the definitions (TF-IDF)

<span style="color: blue;"> Instead of hand-picking keywords, compare each passage to the *full text of each definition*. TF-IDF turns every text into a vector of word weights. Words that are rare across the collection count more. **Cosine similarity** then measures how much two texts overlap in vocabulary (0 = no shared words, 1 = identical).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
vectorizer.fit(list(definitions.definition) + list(passages.text))

D = vectorizer.transform(definitions.definition)
P = vectorizer.transform(passages.text)
sim = pd.DataFrame(cosine_similarity(P, D), index=passages.passage_id, columns=definitions.dimension)

# Best-matching dimension for each passage
top = sim.idxmax(axis=1).rename("tfidf_top_dimension")
best = sim.max(axis=1).round(3).rename("similarity")
pd.concat([passages.set_index("passage_id")[["ticker", "source"]], top, best], axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(sim.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(sim.shape[1]), [c[:22] for c in sim.columns], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(sim.shape[0]), sim.index, fontsize=8)
fig.colorbar(im, label="cosine similarity")
ax.set_title("How much does each passage's wording overlap with each definition?")
plt.tight_layout(); plt.show()

**<span style="color: blue;">**Go deeper:** Similarities are low overall (mostly well below 0.3). Short passages share few exact words with the definitions, even when they clearly describe the concept. Passage M1 says Microsoft "is a major investor in OpenAI", which is plainly corporate venture capital, but those words barely appear in the CVC definition.

**<span style="color: blue;"> 🔧 Try it**: rewrite one definition in plainer language (for example, add "invests in startups" to Corporate Venture Capital), re-run the two cells above, and watch the rankings move. The method measures *wording*, so how you write the definition changes the answer!

In [ ]:
# 🔧 Try it: edit a definition, then re-run the TF-IDF cells above
# definitions.loc[definitions.dimension == "Corporate Venture Capital", "definition"] += " The company is an investor in startups such as OpenAI."

## 4. Coding with an LLM

<span style="color: blue;">An LLM can read for meaning: it can tell that "a major investor in OpenAI" is venture capital and that "AI accelerator options" are chips. To use it for research, we treat it like a research assistant with a **codebook**:

1. Give it the definitions (the codebook).
2. Ask a narrow question with a fixed output format (JSON), so the answers can be processed by code.
3. Require a **verbatim quote** for every claim, so a human can check it.
4. Use `temperature = 0` to reduce randomness.

This is the same design the research pipeline uses, simplified to one passage at a time.

### <span style="color: green;">Q: What do LLMs bring to the table here?

<span style="color: blue;">**A:** Parts 2 and 3 matched *strings*: the same letters in the passage and in a keyword list or definition. An LLM, a large language model trained on enormous amounts of text, can apply *meaning*. For this research question that adds five things:

| What the LLM adds | Example from our passages |
|---|---|
| **Reads words in context** | It can tell that "AI accelerator options" (G5) are chips and "accelerated depreciation" (R2) is accounting, while "a nine-month accelerator" (R5) is a startup program. |
| **Recognizes a concept without its keywords** | "Microsoft is a major investor in OpenAI" (M1) is corporate venture capital, although the words "venture capital" never appear. |
| **Applies a codebook in plain English, with no training data** | We give it our 15 definitions, much as we would brief a research assistant. Changing a definition means editing a sentence, not rebuilding a statistical model. |
| **Returns structured, explained, checkable answers** | Several dimensions per passage, a strength rating, a one-line reason, and a verbatim quote we can verify automatically (see "Trust, but verify" below). |
| **Scales** | The full study codes about 60 companies × 10 or more windows × 15 dimensions. That is weeks of research-assistant time, but hours and a modest budget for an LLM. |

**What it does *not* bring:** guaranteed correctness. LLMs can be confidently wrong or invent evidence ("hallucinate"). Their answers can change with the wording of the prompt, from run to run, and when the model version changes. They can only judge what is in the text they are shown. So LLM output is treated as **measurement that must be validated**: check the quotes, compare with human coders (Part 5), and repeat runs (Part 6).

### <span style="color: green;">Q: What is a prompt?

<span style="color: blue;">**A:** A prompt is everything we send the model: the instructions *and* the text to analyze. The model's answer depends heavily on it, so in research **the prompt is part of your method**, like the wording of a survey question. Report it, keep it fixed across observations, and test how sensitive results are to it.

Our prompt has six parts. You will see each one in the code below:

| Part | In our code | Job |
|---|---|---|
| **System prompt** | `SYSTEM_PROMPT` | Sets the role ("careful research assistant") and ground rules ("return only valid JSON"). |
| **Codebook** | the 15 `definitions` | Gives the model the same definitions a human coder would use. |
| **Input** | the `<passage>` block | The data to code, clearly marked off from the instructions. |
| **Task** | numbered steps 1 to 4 | The exact decisions to make, including what does *not* count. |
| **Output format** | the JSON template | Makes answers machine-readable, so code can tabulate and check them. |
| **Guardrails** | "copy EXACTLY", "Do not guess", "Use only dimension names from the codebook" | Reduce made-up quotes and invented categories. |

Some settings sit outside the text but matter just as much: which **model** is used, **temperature** (0 = least random), and the maximum length of the answer.

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful research assistant coding corporate disclosures for an academic "
    "study of corporate venturing (how established companies engage with startups). "
    "Return only valid JSON, with no other text."
)

def build_prompt(passage_text, definitions=definitions):
    codebook = "\n".join(f"- {r.dimension} ({r.group}): {r.definition}" for _, r in definitions.iterrows())
    return f'''CODEBOOK
{codebook}

PASSAGE
<passage>
{passage_text}
</passage>

TASK
1. Decide whether this passage describes corporate venturing, meaning the company's engagement with
   startups, early-stage ventures, or new ventures it builds. Ordinary operations, accounting terms,
   and internal R&D alone do not count.
2. List every codebook dimension the passage provides evidence for. Use only dimension names from the codebook.
3. For each, copy a short quote (at most 30 words) EXACTLY as it appears in the passage, and rate the
   evidence as "strong", "moderate", or "weak".
4. If there is no evidence, return an empty list. Do not guess.

Return JSON in exactly this shape:
{{"is_corporate_venturing": true or false,
  "dimensions": [{{"dimension": "...", "evidence_quote": "...", "strength": "..."}}],
  "note": "one sentence explaining your judgment"}}'''

print(build_prompt(passages.text[0])[-1500:])   # the end of the prompt for passage G1

In [ ]:
# Take the prompt apart: what does the model actually receive for one passage?
example = build_prompt(passages.text[0])          # passage G1

print("=== SYSTEM PROMPT (sent separately: the model's role and ground rules) ===")
show(SYSTEM_PROMPT)
for part in re.split(r"\n(?=(?:PASSAGE|TASK)\n)", example):
    header, body = part.split("\n", 1)
    print(f"\n=== {header} ({len(body):,} characters) ===")
    print(body[:500] + (" ..." if len(body) > 500 else ""))

print(f"\nThe whole prompt is about {len(example)/4:,.0f} tokens. Most of it is the codebook, "
      "which is identical for every passage.")

<span style="color: blue;">**🔧 Try it:** In the output above, find the three lines of the prompt that are there to prevent hallucination (made-up evidence). Then look back at the dictionary in Part 2: which parts of this prompt play the role that the keyword lists played there?

### Calling the model, or using saved answers (we are using saved answers here)

`USE_API = False` (the default) loads answers saved when this notebook was built. To call the model live you need the `anthropic` package and an API key: add the key in the 🔑 cell below, then set `USE_API = True`.

 <span style="color: green;">Remember from our setup:  **No API key needed.** Part 4 uses saved LLM answers by default. If you were to do this with a live Anthropic API key, set `USE_API = True` to call the model live. We are NOT doing this right now.

### 🔑 <span style="color: green;">Where your API key goes (optional, not used in class)

<span style="color: blue;">We are **not** using an API key today. If you get one later (from the Anthropic Console, console.anthropic.com), this is the only place it goes:

1. In a terminal, run `pip install anthropic`.
2. Put your key in the 🔑 code cell just below, using **Option A** or **Option B**, and run it.
3. In the next cell, change `USE_API = False` to `USE_API = True`, then re-run it.

**Keep your key private.** It works like a password and is billed to your account. If you paste it into the notebook (Option A), delete it before you share, email or upload the file. Option B asks for the key in a hidden box and never saves it in the notebook.

In [ ]:
# 🔑 ================= YOUR API KEY GOES HERE (optional) =================
import os
from getpass import getpass

API_KEY = ""     # Option A: paste your key between the quotes, e.g. API_KEY = "sk-ant-..."
                 #           Delete it again before sharing this notebook!

# API_KEY = getpass("Paste your Anthropic API key, then press Enter: ")   # Option B (safer): remove the # at the start of this line

if API_KEY.strip():
    os.environ["ANTHROPIC_API_KEY"] = API_KEY.strip()

if os.environ.get("ANTHROPIC_API_KEY"):
    print("✓ API key found. To call the model live, set USE_API = True in the next cell.")
else:
    print("No API key set. That's fine: the notebook will use the saved answers.")
# =====================================================================

In [ ]:
import os

USE_API = False                 # 🔧 set to True ONLY after adding your key in the 🔑 cell above
MODEL   = "claude-haiku-5-5"    # a small, inexpensive model; try a larger one and compare

def parse_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)   # strip a code fence if present
    return json.loads(text)

def llm_code(passage_id, prompt_builder=build_prompt):
    text = passages.set_index("passage_id").loc[passage_id, "text"]
    if not USE_API:
        return cached_llm[passage_id]
    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("USE_API is True but no API key is set. Add your key in the 🔑 cell above, or set USE_API = False.")
    import anthropic
    client = anthropic.Anthropic()          # uses the key set in the 🔑 cell (ANTHROPIC_API_KEY)
    msg = client.messages.create(
        model=MODEL, max_tokens=800, temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt_builder(text)}],
    )
    return parse_json(msg.content[0].text)

llm_results = {pid: llm_code(pid) for pid in passages.passage_id}
print(json.dumps(llm_results["M1"], indent=2))

In [ ]:
rows = []
for pid, res in llm_results.items():
    for d in res["dimensions"] or [{"dimension": "(none)", "evidence_quote": "", "strength": ""}]:
        rows.append({"passage_id": pid, "venturing?": res["is_corporate_venturing"], **d})
llm_table = pd.DataFrame(rows)
llm_table

### Trust, but verify: are the quotes real?

LLMs can **hallucinate**: produce text that sounds right but is not in the source. Because we asked for verbatim quotes, we can check every one automatically. The research pipeline does a version of this too.

In [ ]:
def quote_is_verbatim(row):
    if row["dimension"] == "(none)":
        return None
    source = passages.set_index("passage_id").loc[row["passage_id"], "text"]
    return row["evidence_quote"] in source

llm_table["quote_found_in_text"] = llm_table.apply(quote_is_verbatim, axis=1)
print(llm_table.quote_found_in_text.value_counts(dropna=True))
llm_table[llm_table.quote_found_in_text == False]

**<span style="color: blue;">**Go deeper:**
* Compare the LLM's answers for G5, M5, M6 and R2 with the dictionary results from Part 2. What did the LLM understand that the keyword count did not?
* Look at G3 (Google Cloud) and M2 (Microsoft's ecosystem). The LLM coded a dimension but said "not corporate venturing". Is that a contradiction, or a sensible distinction?

**<span style="color: blue;">🔧 Future work if interested: Try it if interested, but you would need an API key ** (needs `USE_API = True`)
1. Remove the codebook from the prompt (write a `build_prompt_no_codebook` function) and re-run M1 and R1. What changes?
2. Run the same passage three times with `temperature=1`. Do the answers stay the same?
3. Switch `MODEL` to a larger model and compare the answers for the ambiguous passages (G3, M4, R1).

## 5. You are the human coder

<span style="color: blue;">The real test of any method is agreement with careful human judgment. Code a few passages yourself **before** looking at the methods' answers again. Use dimension names exactly as they appear in the codebook.

In [ ]:
# 🔧 Try it: fill in your own codes (an empty list means "no evidence")
# Copy dimension names exactly from the codebook, e.g. ["Corporate Venture Capital", "Mentorship"]
my_codes = {
    "G1": [],
    "G6": [],
    "M1": [],
    "M7": [],
    "R1": [],
    "R6": [],
}

In [ ]:
llm_dims = {pid: sorted(d["dimension"] for d in r["dimensions"]) for pid, r in llm_results.items()}
kw_dims  = passages.set_index("passage_id").keyword_dims.to_dict()
tfidf_top = sim.idxmax(axis=1).to_dict()

def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not a and not b else len(a & b) / len(a | b)

compare = pd.DataFrame([{
    "passage": pid,
    "you": ", ".join(codes) or "(none)",
    "keywords": ", ".join(kw_dims[pid]) or "(none)",
    "tf-idf top": tfidf_top[pid],
    "LLM": ", ".join(llm_dims[pid]) or "(none)",
    "agree you-LLM": round(jaccard(codes, llm_dims[pid]), 2),
    "agree you-keywords": round(jaccard(codes, kw_dims[pid]), 2),
} for pid, codes in my_codes.items()])
compare

<span style="color: blue;">The agreement columns use the **Jaccard index**: shared codes divided by all codes either side used (1 = identical, 0 = nothing in common).

**<span style="color: blue;">Go Deeper:** Where you disagree with the LLM, who is right? Is the codebook clear enough that two humans would agree? In research this is measured as *inter-coder reliability*, and it applies to people and to models.

## <span style="color: green;">Answer key (suggested)

<span style="color: blue;">Fill in your own codes **before** reading this. Below is how I would code the passages, and why. Coding involves judgment: reasonable coders can disagree on the weak cases, and those disagreements are part of the lesson.

| Passage | Suggested codes | Why |
|---|---|---|
| **G1** Alphabet, Other Bets (10-K) | Venture Building; Accelerators and Incubators; Market Expansion; Access to New Technology | Other Bets "operate as independent companies" with their own boards and outside investors (building ventures). X, the "moonshot factory", is an internal incubator, and our codebook names X explicitly. Waymo is "entering international markets". X is "developing breakthrough technologies". |
| **G6** Google for Startups Accelerator (website) | Accelerators and Incubators; Mentorship; Events | A named program with cohorts of 10 to 15 startups. Founders are "paired with relevant experts". Group learning sessions and sprint projects are weaker evidence of events. Ignore the stray "LIV UP..." line, which is scraped carousel text. |
| **M1** Microsoft and OpenAI (10-K) | Corporate Venture Capital; Venture Clienting; Access to New Technology | Microsoft "is a major investor in OpenAI" (equity stake). The "reciprocal revenue-sharing arrangements" make it a commercial partner too. It holds "rights to OpenAI's intellectual property ... for integration into our products". *Debate:* is OpenAI still a startup, and does that matter? |
| **M7** M12, Microsoft's venture fund (website) | Corporate Venture Capital; Access to New Technology | A named CVC fund that invests "in promising seed through B-stage companies", with focus areas in foundational technologies. Ignore the "Lorem Ipsum" placeholder text the scraper picked up. |
| **R1** Merck, externally sourced programs (10-K) | Venture Clienting; Access to New Technology | Licensing, alliances and acquisitions "from early research to late-stage compounds", explicitly "as well as access to new technologies". **Not** CVC: no equity investment is mentioned. |
| **R6** Merck Digital Sciences Studio (website) | Mentorship; Events; Corporate Venture Capital; Develop Business Ecosystems | Mentors, structured workshop sessions, and a "$100 000 USD SAFE investment" (an early-stage equity instrument). The "supportive cohort of founders, alumni, and corporate innovators" is weaker evidence of ecosystem building. |

The code cell below holds the full key for all 19 passages and scores every method against it.

In [ ]:
# Suggested answer key for all 19 passages (an empty list means no corporate-venturing evidence)
ANSWER_KEY = {
    "G1": ["Venture Building", "Accelerators and Incubators", "Market Expansion", "Access to New Technology"],
    "G2": ["Corporate Venture Capital", "Financial Return"],      # non-marketable equity = stakes in private companies
    "G3": [],                                                      # core cloud business; Business Services is arguable
    "G4": ["Developing Entrepreneurial Culture and Capabilities"],
    "G5": [],                                                      # "AI accelerator" = computer chips
    "G6": ["Accelerators and Incubators", "Mentorship", "Events"],
    "M1": ["Corporate Venture Capital", "Venture Clienting", "Access to New Technology"],
    "M2": ["Develop Business Ecosystems"],
    "M3": [],                                                      # sustainability of own operations, not venturing
    "M4": ["Corporate Venture Capital"],                           # weak: total may include non-venture holdings
    "M5": [],                                                      # "business climate" = accounting boilerplate
    "M6": [],                                                      # "accelerated method" = stock compensation
    "M7": ["Corporate Venture Capital", "Access to New Technology"],
    "R1": ["Venture Clienting", "Access to New Technology"],
    "R2": [],                                                      # "accelerated depreciation"
    "R3": [],                                                      # market expansion, but not through venturing
    "R4": [],                                                      # internal R&D
    "R5": ["Accelerators and Incubators", "Corporate Venture Capital", "Mentorship"],
    "R6": ["Mentorship", "Events", "Corporate Venture Capital", "Develop Business Ecosystems"],
}

# How close is each method (and you) to the key? Jaccard: 1 = same codes, 0 = nothing in common
key_compare = pd.DataFrame([{
    "passage": pid,
    "key": ", ".join(codes) or "(none)",
    "keywords": round(jaccard(codes, kw_dims[pid]), 2),
    "tf-idf top": round(jaccard(codes, [tfidf_top[pid]]), 2),
    "LLM": round(jaccard(codes, llm_dims[pid]), 2),
    "you": round(jaccard(codes, my_codes[pid]), 2) if pid in my_codes else None,
} for pid, codes in ANSWER_KEY.items()])

print("Average agreement with the key (all 19 passages):")
print(key_compare[["keywords", "tf-idf top", "LLM"]].mean().round(2).to_string())
if any(my_codes.values()):
    print("\nYour average agreement on the passages you coded:", round(key_compare["you"].dropna().mean(), 2))
else:
    print("\nYou have not filled in my_codes yet: add your codes in the 🔧 cell above and re-run.")
key_compare

<span style="color: blue;">**Go deeper:** Look at the rows where the LLM scores below 1. On G1, the saved LLM answer left out *Access to New Technology*; on G3, M3 and R3 it coded a weak dimension where the key says "none". Are those LLM errors, or is the key debatable? Notice also that TF-IDF always returns *some* top dimension, so it can never say "none". That is one reason it scores poorly on the trap passages.

## 6. From evidence to company scores

<span style="color: blue;">The full research pipeline reads every window of each 10-K plus the company's venture web pages, collects evidence for all 15 dimensions, and then asks the LLM to assign each dimension an **emphasis score from 0 to 100** against a written rubric. Below are its actual results for six companies.

In [ ]:
S = scores.set_index("ticker").drop(columns="company")
fig, ax = plt.subplots(figsize=(12, 4.2))
im = ax.imshow(S.values, cmap="viridis", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(S.shape[1]), [c[:24] for c in S.columns], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(S.shape[0]), S.index)
for i in range(S.shape[0]):
    for j in range(S.shape[1]):
        v = S.values[i, j]
        ax.text(j, i, v, ha="center", va="center", fontsize=7, color="white" if v < 60 else "black")
for x in (3.5, 7.5):
    ax.axvline(x, color="white", lw=2)
ax.set_title("Emphasis scores (0-100): Programs | Activities | Objectives")
fig.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# Look inside one score: the quote and rationale the model gave
# 🔧 Try it: change the company (GOOGL or MRK) and dimension
card = next(c for c in score_cards if c["ticker"] == "GOOGL" and c["dimension"] == "Workspace")
print(f'{card["ticker"]} · {card["dimension"]}: score {card["score"]} (rubric band {card["rubric_band"]})\n')
print("Top quote:"); show(card["top_quote"]); print("\nRationale:"); show(card["rationale"])

**<span style="color: blue;">Go Deeper:** Google scores 15 on *Workspace* even though it runs Google for Startups Campus locations. The rationale explains why: the documents the pipeline read did not mention them. **Scores measure emphasis in the disclosures, not reality.** What is missing from the data is invisible to any method.

### Reliability: ask the same question five times

<span style="color: blue;">Even at `temperature = 0`, LLM outputs can vary from call to call. The pipeline therefore scored each company **five times** and reports the median. The spread shows how much to trust a score.

In [ ]:
rel = pd.DataFrame([{
    "ticker": c["ticker"], "dimension": c["dimension"], "median": c["score"],
    "min": min(c["ensemble_samples"]), "max": max(c["ensemble_samples"]),
    "range": max(c["ensemble_samples"]) - min(c["ensemble_samples"]),
} for c in score_cards if c["ensemble_samples"]])
rel.sort_values("range", ascending=False).head(8)

**🔧 <span style="color: blue;">Try it: build your own index**

Suppose you want one number for how strongly a company invests in startups *for strategic reasons* rather than *for money*. Define it from the scores, rank the six companies, and defend your formula.

In [ ]:
# 🔧 Try it: change the formula
strategic = S[["Access to New Technology", "Develop Business Ecosystems", "Market Expansion"]].mean(axis=1)
financial = S["Financial Return"]
index = (strategic - financial).rename("strategic minus financial")
index.sort_values(ascending=False).plot.barh(title="Strategic vs. financial orientation (your index)")
plt.xlabel("points"); plt.tight_layout(); plt.show()

## <span style="color: green;"> Wrap-up: questions to take away

1. **Validity.** Does a high "emphasis score" mean a company is *good* at corporate venturing, or only that it *talks* about it? What other data would you want?
2. **Source matters.** 10-Ks are written for investors and leave out small programs. Websites are promotional. How would you combine them, and should their evidence count the same?
3. **Method choice.** When would you still use a dictionary instead of an LLM? Think about cost, speed, transparency, and replicability.
4. **Checking the machine.** Which safeguards in this notebook (verbatim quotes, fixed output format, temperature 0, repeated runs, human coding) would you keep in your own project?

---
*<span style="color: green;"> Data: excerpts from Alphabet, Microsoft and Merck 10-K filings (fiscal 2025) and company websites, with scores from the Corporate Venturing research toolkit, © 2026 Sheryl Winston Smith. For teaching use.*